## Trim messages
https://python.langchain.com/docs/how_to/trim_messages/#using-with-chatmessagehistory

If passing the trimmed chat history back into a chat model directly, the trimmed chat history should satisfy the following properties:

1. The resulting chat history should be valid. Usually this means that the following properties should be satisfied:
    * The chat history starts with either (1) a HumanMessage or (2) a SystemMessage followed by a HumanMessage.
    
    * The chat history ends with either a HumanMessage or a ToolMessage.
    * A ToolMessage can only appear after an AIMessage that involved a tool call. This can be achieved by setting start_on="human" and ends_on=("human", "tool").
2. It includes recent messages and drops old messages in the chat history. This can be achieved by setting strategy="last".

3. Usually, the new chat history should include the SystemMessage if it was present in the original chat history since the SystemMessage includes special instructions to the chat model. The SystemMessage is almost always the first message in the history if present. This can be achieved by setting include_system=True.

In [ ]:
!pip show langchain

Name: langchain
Version: 0.3.13
Summary: Building applications with LLMs through composability
Home-page: https://github.com/langchain-ai/langchain
Author: 
Author-email: 
License: MIT
Location: /Users/manuelalejandroquesada/miniconda3/envs/langchain_env/lib/python3.12/site-packages
Requires: aiohttp, langchain-core, langchain-text-splitters, langsmith, numpy, pydantic, PyYAML, requests, SQLAlchemy, tenacity
Required-by: 


In [40]:
# ! pip install langchain-core langgraph>0.2.27 ollama python-dotenv


from langchain_ollama.chat_models import ChatOllama
from dotenv import load_dotenv, find_dotenv
import os
import warnings
from IPython.display import display, Markdown  # to see better the output text

warnings.filterwarnings("ignore")
_ = load_dotenv(find_dotenv())  # read local .env file

os.environ["LANGCHAIN_TRACING_V2"] = "true"

# ! pip install langchain_ollama

llm = ChatOllama(
    model="gemma2:latest",
    temperature=0.1,
)

In [6]:
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage,
    ToolMessage,
    trim_messages,
)
from langchain_openai import ChatOpenAI

messages = [
    SystemMessage("you're a good assistant, you always respond with a joke."),
    HumanMessage("i wonder why it's called langchain"),
    AIMessage(
        'Well, I guess they thought "WordRope" and "SentenceString" just didn\'t have the same ring to it!'
    ),
    HumanMessage("and who is harrison chasing anyways"),
    AIMessage(
        "Hmmm let me think.\n\nWhy, he's probably chasing after the last cup of coffee in the office!"
    ),
    HumanMessage("what do you call a speechless parrot"),
]


trim_messages(
    messages,
    # Keep the last <= n_count tokens of the messages.
    strategy="last",
    # Remember to adjust based on your model or else pass a custom
    # token_encoder or `len` to count the number of message instead of tokens.
    token_counter=llm,

    # Remember to adjust based on the desired conversation length
    max_tokens=80,
    # Most chat models expect that chat history starts with either:
    # (1) a HumanMessage or
    # (2) a SystemMessage followed by a HumanMessage
    start_on="human",
    # Most chat models expect that chat history ends with either:
    # (1) a HumanMessage or
    # (2) a ToolMessage
    end_on=("human", "tool"),
    # Usually, we want to keep the SystemMessage
    # if it's present in the original history.
    include_system=True,
    allow_partial=False,
)

[SystemMessage(content="you're a good assistant, you always respond with a joke.", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='and who is harrison chasing anyways', additional_kwargs={}, response_metadata={}),
 AIMessage(content="Hmmm let me think.\n\nWhy, he's probably chasing after the last cup of coffee in the office!", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='what do you call a speechless parrot', additional_kwargs={}, response_metadata={})]

## Trimmer as chaining
```trim_messages``` can be used imperatively (like above) or declaratively, making it easy to compose with other components in a chain

In [15]:
# Notice we don't pass in messages. This creates
# a RunnableLambda that takes messages as input
trimmer = trim_messages(
    token_counter=len,
    strategy="last",
    max_tokens=5,
    start_on="human",
    end_on=("human", "tool"),
    include_system=True,
)

chain = trimmer | llm
chain.invoke(messages, config={"configurable": {
             "run_id": "abc123"}}).pretty_print()

================================== Ai Message ==================================

A Polly want a cracker... but can't say it! 🦜  


Let me know if you want to hear another joke! 😄


 ### Important: Use ```run_id``` or another metadata for tracking the chain on LangSmith

## Build a custom Chat Message History

In [146]:
from operator import itemgetter
from typing import List


from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from pydantic import BaseModel, Field
from langchain_core.runnables import (
    RunnableLambda,
    ConfigurableFieldSpec,
    RunnablePassthrough,
)


class InMemoryHistory(BaseChatMessageHistory, BaseModel):
    """In memory implementation of chat message history."""

    messages: List[BaseMessage] = Field(default_factory=list)

    def add_messages(self, messages: List[BaseMessage]) -> None:
        """Add a list of messages to the store"""
        self.messages.extend(messages)

    def clear(self) -> None:
        self.messages = []

# Here we use a global variable to store the chat message history.
# This will make it easier to inspect it to see the underlying results.
store = {}

def get_by_session_id(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryHistory()
    return store[session_id]


history = get_by_session_id("1")
history.add_message(AIMessage(content="hello"))
print(store)

{'1': InMemoryHistory(messages=[AIMessage(content='hello', additional_kwargs={}, response_metadata={})])}


In [147]:
from typing import Optional

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory


chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You're an assistant who's good at {ability}"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}"),
])

chain = chat_prompt | llm

chain_with_history = RunnableWithMessageHistory(
    chain,
    # Uses the get_by_session_id function defined in the example
    # above.
    get_by_session_id,
    input_messages_key="question",
    # this name must match the placeholder name in the prompt
    history_messages_key="history",
)

Markdown(chain_with_history.invoke(
    {"ability": "math", "question": "What does cosine mean?"},
    config={"configurable": {"session_id": "foo"}}
).content)

Cosine is a trigonometric function that relates the angle of a right triangle to the ratio of two sides. 

Here's a breakdown:

* **Right Triangle:** Cosine works with right triangles, which have one angle measuring 90 degrees.
* **Angle:**  You choose an angle in the right triangle (let's call it θ).
* **Sides:**
    * **Adjacent Side:** The side next to the chosen angle (θ), but not the hypotenuse.
    * **Hypotenuse:** The longest side of the triangle, opposite the right angle.

**Definition:** Cosine (cos) of an angle θ is defined as:

  cos(θ) = Adjacent Side / Hypotenuse


**In simpler words:** Cosine tells you how long the adjacent side is compared to the hypotenuse for a given angle in a right triangle. 

Let me know if you'd like to see a diagram or have any other questions about cosine!

In [148]:
Markdown(chain_with_history.invoke(  # noqa: T201
    {"ability": "math", "question": "What's its inverse"},
    config={"configurable": {"session_id": "foo"}}
).content)

The inverse of cosine is called **arccosine**, often written as **cos⁻¹(x)** or **acos(x)**.  

Here's what it does:

* **Input:** You give arccosine a ratio (between -1 and 1).
* **Output:** It gives you the angle (in radians or degrees) whose cosine is that ratio.

**Example:**

If cos(30°) = √3/2, then arccos(√3/2) = 30°.


Essentially, arccosine "undoes" what cosine does.

### See all the history

As we can see ```RunnableWithMessageHistory```ensure that the query and AI responses are added int the ChatMessageHistory

In [149]:
print(store["foo"])

Human: What does cosine mean?
AI: Cosine is a trigonometric function that relates the angle of a right triangle to the ratio of two sides. 

Here's a breakdown:

* **Right Triangle:** Cosine works with right triangles, which have one angle measuring 90 degrees.
* **Angle:**  You choose an angle in the right triangle (let's call it θ).
* **Sides:**
    * **Adjacent Side:** The side next to the chosen angle (θ), but not the hypotenuse.
    * **Hypotenuse:** The longest side of the triangle, opposite the right angle.

**Definition:** Cosine (cos) of an angle θ is defined as:

  cos(θ) = Adjacent Side / Hypotenuse


**In simpler words:** Cosine tells you how long the adjacent side is compared to the hypotenuse for a given angle in a right triangle. 

Let me know if you'd like to see a diagram or have any other questions about cosine!
Human: What's its inverse
AI: The inverse of cosine is called **arccosine**, often written as **cos⁻¹(x)** or **acos(x)**.  

Here's what it does:

* **Input:

## Using Trimmer  with ChatMessageHistory

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.runnables import RunnableAssign, RunnableParallel

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You're an assistant who's good at {ability}"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}"),
])


trimmer = trim_messages(
    max_tokens=5,  # in this case we want to keep the last 5 messages
    strategy="last",
    token_counter=len,
    include_system=True,
    start_on="human",
    end_on=("human", "tool"),
)

# in this case we need to use `chat_prompt` instead of `history`
# because the history doesn't have the system message
chain_with_history = RunnableWithMessageHistory(
    RunnableAssign(RunnableParallel()).assign(
        chat_prompt=chat_prompt).assign(trimmer=itemgetter(
            "chat_prompt") | trimmer).assign(
                ai_answer=itemgetter("trimmer") | llm),
    get_by_session_id,
    input_messages_key="question",
    history_messages_key="history",
    output_messages_key="ai_answer", # to store the answer in chat history
)

# chain =  RunnableAssign(RunnableParallel()).assign(history=chain_with_history)  #chat_prompt | trimmer | llm,

chain_with_history.invoke(
    {"ability": "math",
     "question": "Tell me the mathematical formula for the previous concepts"},
    config={"configurable": {"session_id": "foo"}},
)

{'ability': 'math',
 'question': 'Tell me the mathematical formula for the previous concepts',
 'history': [HumanMessage(content='What does cosine mean?', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Cosine is a trigonometric function that relates the angle of a right triangle to the ratio of two sides. \n\nHere's a breakdown:\n\n* **Right Triangle:** Cosine works with right triangles, which have one angle measuring 90 degrees.\n* **Angle:**  You choose an angle in the right triangle (let's call it θ).\n* **Sides:**\n    * **Adjacent Side:** The side next to the chosen angle (θ), but not the hypotenuse.\n    * **Hypotenuse:** The longest side of the triangle, opposite the right angle.\n\n**Definition:** Cosine (cos) of an angle θ is defined as:\n\n  cos(θ) = Adjacent Side / Hypotenuse\n\n\n**In simpler words:** Cosine tells you how long the adjacent side is compared to the hypotenuse for a given angle in a right triangle. \n\nLet me know if you'd like to see a diag

It just used the previous answer about inverse cosine cause the trimmer just allow 5 previuos messages, in this case:
1. The system

2. The last Human message
3. The last AI message
4. Actual Question


![alt](resources/after_trim.png)

### You can see as the question and the answer have been added into the ChatHistory

In [152]:
print(store["foo"].messages[-2].content)
Markdown(store["foo"].messages[-1].content)

Tell me the mathematical formula for the previous concepts


You're asking about the formula for the inverse cosine function (arccosine).  

Here it is:

* **y = arccos(x)** 

This means:

* **y** represents the angle in radians.
* **x** represents the ratio of the adjacent side to the hypotenuse in a right triangle (between -1 and 1 inclusive).


Let me know if you'd like to explore any specific examples or applications of arccosine!